In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import pandas as pd
from pathlib import Path

In [3]:
df_raw = pd.read_csv("../data/raw/questions.csv")
print(f"Shape: {df_raw.shape} \nColumns: {list(df_raw.columns)}")

df_raw.head()

Shape: (404351, 6) 
Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


# Drop and Missing Values

## Columns removed: `id`, `qid1`, `qid2`

- `id` is only a row number: it carries no information about the questions.
- `qid1` and `qid2` do not identify a question reliably. The same question text appears under many different ids, and the number of distinct ids is far larger than the number of distinct texts (see the output below). Counting or grouping by them would split one question into many.

We build our own question ids from the normalised text instead. The original columns stay available in `df_raw`.

Moreover, as we can see from the cell below the number of null values is negletable compared to the number of rows (only $3$), hence they are dropped as well.

The small dataset containing the Null values is memorized in case is necessary. After dropping the rows the index is reseted, so the index doesn't contain gaps.


In [4]:
def stack_questions(df_raw):
    q1 = df_raw[ ["qid1", "question1"] ]
    q2 = df_raw[ ["qid2", "question2"] ]

    q1 = q1.rename(columns={"qid1": "qid", "question1": "question"})
    q2 = q2.rename(columns={"qid2": "qid", "question2": "question"})

    q1_with_q2 = pd.concat([q1, q2]).dropna()
    return q1_with_q2

def report_id_problems(df_raw):
    """Print why id, qid1 and qid2 are not useful."""
    q1_with_q2 = stack_questions(df_raw)
    ids_per_text = q1_with_q2.groupby("question")["qid"].nunique()
    texts_per_id = q1_with_q2.groupby("qid")["question"].nunique()

    print(f"Distinct qids: {q1_with_q2['qid'].nunique()}")
    print(f"Distinct question: {q1_with_q2['question'].nunique()}")
    
    print("\nQuestions with most different ids:")
    print(ids_per_text.sort_values(ascending=False).head())


report_id_problems(df_raw)

Distinct qids: 789798
Distinct question: 537387

Questions with most different ids:
question
What are the best ways to lose weight?                                             161
How can you look at someone's private Instagram account without following them?    120
How can I lose weight quickly?                                                     111
What's the easiest way to make money online?                                        88
Can you see who views your Instagram?                                               79
Name: qid, dtype: int64


In [5]:
def drop_unused_columns(df_raw):
    """Drop columns id, qid1 and qid2."""
    return df_raw.drop(columns=["id", "qid1", "qid2"])

df = drop_unused_columns(df_raw)

def drop_missing(df):
    """Drop rows with a missing value. 
    Returns the clean table AND the dropped rows."""
    df = df.copy()  
    df_null = df[df.isna().any(axis=1)]

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df, df_null


print("Number of Null Values Before")
print(df.isna().sum())
print(f"\nShape before: {df.shape}")

df, df_null = drop_missing(df)

print("\nNumber of Null Values After")
print(f"Shape after: {df.shape}\n")
print(df.isna().sum())
df_null.head()

Number of Null Values Before
question1       1
question2       2
is_duplicate    0
dtype: int64

Shape before: (404351, 3)

Number of Null Values After
Shape after: (404348, 3)

question1       0
question2       0
is_duplicate    0
dtype: int64


,question1,question2,is_duplicate
105796,How can I develop android app?,NaN,0
201871,How can I create an Android app?,NaN,0
363416,NaN,My Chinese name is Haichao Yu. What English na...,0


In [6]:
# 1 Question ids: one integer per distinct text_norm, and pairs rewritten with those ids.
# 2 Pair checks: remove pairs where both sides are the same question, remove repeated pairs (smaller id first), and handle pairs that have both labels
# 3 Ground-truth groups: connected components of the pairs labelled duplicate, with the group size distribution.
# 4 Tuning and test split: about 20% / 80% of the questions, split by connected component of the whole pair graph and with a fixed seed, so that no question appears on both sides.
# 5 Save: questions.parquet (id, text_raw, text_norm, group, split), pairs.parquet (ids and is_duplicate), checked by reading them back.
# 6 Numbers for the report: class balance (36.9% duplicates), how many pairs each question appears in, and every count printed above.

# Normalization
Light normalisation: NFKC, lower case, whitespace collapsed. This only makes identical questions look identical, so heavy cleaning (stop words, punctuation) stays in P2’s experiments. After it we give each distinct text one qid, and from then on pairs refer to qids, not texts.

In [7]:
def normalise_questions(df):
    """Adds question1_norm / question2_norm"""
    df = df.copy()
    df["question1_norm"] = df["question1"]
    df["question2_norm"] = df["question2"]

    for col in ["question1_norm", "question2_norm"]:
        df[col] = df[col].str.normalize("NFKC")
        df[col] = df[col].str.lower()
        df[col] = df[col].str.split().str.join(" ")

    target = df.pop("is_duplicate")
    df["is_duplicate"] = target
    return df

df = normalise_questions(df)
df.head()

,question1,question2,question1_norm,question2_norm,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,what is the story of kohinoor (koh-i-noor) dia...,what would happen if the indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,why am i mentally very lonely? how can i solve...,find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,"which one dissolve in water quikly sugar, salt...",which fish would survive in salt water?,0


In [8]:
def build_questions_table(df):
    """One row per distinct normalised question.
    The row index is the qid."""
    q1_norm = df["question1_norm"].rename("question")
    q2_norm = df["question2_norm"].rename("question")

    df_unique_questions = pd.concat([q1_norm, q2_norm], ignore_index=True).drop_duplicates().reset_index(drop=True)
    return df_unique_questions

df_unique_questions = build_questions_table(df)
df_unique_questions.head(10)

0    what is the step by step guide to invest in sh...
1    what is the story of kohinoor (koh-i-noor) dia...
2    how can i increase the speed of my internet co...
3    why am i mentally very lonely? how can i solve...
4    which one dissolve in water quikly sugar, salt...
5    astrology: i am a capricorn sun cap moon and c...
6                                  should i buy tiago?
7                       how can i be a good geologist?
8                      when do you use シ instead of し?
9    motorola (company): can i hack my charter moto...
Name: question, dtype: object

In [9]:
def build_pairs_table(df, df_unique_questions):
    """Replaces each question text by its qid."""
    unique_ids = df_unique_questions.to_dict()
    unique_ids = dict((v, k) for k, v in unique_ids.items())

    df_pair_questions = pd.DataFrame({"qid1": df["question1_norm"].map(unique_ids),
                                      "qid2": df["question2_norm"].map(unique_ids),
                                      "is_duplicate": df["is_duplicate"]})
    return df_pair_questions, unique_ids

df_pair_questions, unique_ids = build_pairs_table(df, df_unique_questions)
df_pair_questions.head(10)

,qid1,qid2,is_duplicate
0,0,290379,0
1,1,16735,0
2,2,290380,0
3,3,290381,0
4,4,290382,0
5,5,290383,1
6,6,290384,0
7,7,290385,1
8,8,273498,0
9,9,290386,0


# Incorrect Pairs
Three problems can hide in the pairs table, and each is checked separately:
- identical pair: both sides have the same qid (a question paired with itself)
- repeated pair: the same two questions appear in more than one row, in either order (A,B and B,A count as the same)
- inconsistent pair: a repeated pair whose rows carry different labels


In [10]:
def check_pairs(df_pair_questions):
    """Detect identical, repeated and inconsistent pairs"""
    df_questions = df_pair_questions.copy()

    # identical, both sides are the same question
    identical_pairs = df_questions["qid1"] == df_questions["qid2"]
    print(f"Number of Identical Pairs: {identical_pairs.sum()}")

    # repeated, same two questions in more than one row, in either order
    sorted_pairs = df_questions[["qid1", "qid2"]].apply(lambda row: tuple(sorted(row)), axis=1)
    sorted_pairs = sorted_pairs.rename("sorted_pairs")

    repeated_pairs = sorted_pairs.duplicated()
    print(f"Number of Repeated Pairs: {repeated_pairs.sum()}")

    # inconsistent, the same pair appears with different labels
    sorted_pairs = sorted_pairs.to_frame()
    sorted_pairs["is_duplicate"] = df_questions["is_duplicate"]

    inconsistent_pairs = sorted_pairs.groupby("sorted_pairs")["is_duplicate"].transform("nunique") > 1
    print(f"Number of Inconsistent Pairs: {inconsistent_pairs.sum()}")

    return identical_pairs, repeated_pairs, inconsistent_pairs, sorted_pairs["sorted_pairs"]

identical_pairs, repeated_pairs, inconsistent_pairs, sorted_pairs = check_pairs(df_pair_questions)


Number of Identical Pairs: 38
Number of Repeated Pairs: 528
Number of Inconsistent Pairs: 20


In [11]:
def clean_pairs(df_pair_questions, identical_pairs, repeated_pairs, inconsistent_pairs, sorted_pairs):
    """Drops identical, repeated and inconsistent pairs"""
    remove_rows = identical_pairs | repeated_pairs | inconsistent_pairs
    df_clean = df_pair_questions[~remove_rows].copy()

    df_clean[["qid1", "qid2"]] = sorted_pairs.loc[~remove_rows].tolist()
    df_clean = df_clean.reset_index(drop=True)

    print(f"Rows removed: {remove_rows.sum()}, rows kept: {len(df_clean)}")
    return df_clean

df_clean = clean_pairs(df_pair_questions, identical_pairs, repeated_pairs, inconsistent_pairs, sorted_pairs)

Rows removed: 576, rows kept: 403772


# Ground-truth groups

If A is a duplicate of B and B of C, then A and C are duplicates too. To get these groups, we build a graph where the questions are nodes and every pair labelled `is_duplicate = 1` is an edge. Each connected component of this graph is one group of questions that all mean the same thing.

The groups give us two things:
- the duplicate pairs implied by the groups, s(s-1)/2 for a group of size s.
- a check of the labels: a pair labelled 0 whose two questions are in the same group contradicts the other labels. These are flagged as `conflicting_pair`. They are not the inconsistent pairs removed earlier, which were the same pair with two different labels.

In [12]:
def build_groups(df_clean, df_unique_questions):
    """Connected components of the duplicate pairs = ground-truth groups.
    Returns the questions table (with group_id), df_clean with a conflicting_pair flag, and the group sizes."""
    all_duplicates = df_clean[df_clean["is_duplicate"] == 1]
    n = len(df_unique_questions)

    graph = coo_matrix((np.ones(len(all_duplicates)), (all_duplicates["qid1"], all_duplicates["qid2"])), shape=(n, n))
    number_of_groups, group_id = connected_components(graph, directed=False)
    sizes = pd.Series(group_id).value_counts()

    print(f"Number of Groups: {number_of_groups} \nGroups with Size greater than 1: {(sizes > 1).sum()}")
    print(f"Largest group: {sizes.max()} questions")
    print(f"Labelled duplicate pairs: {len(all_duplicates)}")
    print(f"Duplicate pairs implied by the groups: {(sizes * (sizes - 1) // 2).sum()}")

    questions = df_unique_questions.to_frame(name="text_norm")
    questions.index.name = "qid"
    questions = questions.reset_index()
    questions["group_id"] = group_id

    df_clean = df_clean.copy()
    
    df_clean["conflicting_pair"] = (
        (df_clean["is_duplicate"] == 0)
        & (group_id[df_clean["qid1"]] == group_id[df_clean["qid2"]])
    )
    print(f"Number of Conflicting Pairs: {df_clean['conflicting_pair'].sum()}")
    return questions, df_clean, sizes

questions, df_clean, sizes = build_groups(df_clean, df_unique_questions)

Number of Groups: 448100 
Groups with Size greater than 1: 60379
Largest group: 108 questions
Labelled duplicate pairs: 148773
Duplicate pairs implied by the groups: 229223
Number of Conflicting Pairs: 107


In [13]:
gid = sizes[sizes == 5].index[0]
print(questions[questions["group_id"] == gid].text_norm)

9656         is it possible to change eye colour naturally?
43591     can i change my eye colour using subliminal me...
53711     is it possible to lighten my dark brown eyes n...
158412             what can natrully change your eye color?
323104               can subminals audio change eye colour?
Name: text_norm, dtype: object


# Dev / test split

The test set is used once, for the final numbers. Everything else uses dev only.

Splitting by pair would leak information: if A~B is in dev and B~C is in test, the model has already seen B. So whole connected components of all labelled pairs (duplicate or not) go to the same side. Components are picked at random, with a fixed seed (42), until test holds about 20% of the questions; the rest is dev.

The split only fixes which pairs are used for tuning and which for the final score. The searches (LSH, HNSW) still run over all questions, because a duplicate in the test set has to be found among all candidates.

In [14]:
def split_dev_test(df_clean, questions, test_fraction=0.2, seed=42):
    """Splits whole connected components of the labelled pairs into dev and test."""
    n = len(questions)

    all_pairs_graph = coo_matrix((np.ones(len(df_clean)), (df_clean["qid1"], df_clean["qid2"])), shape=(n, n))
    number_of_components, component_id = connected_components(all_pairs_graph, directed=False)
    component_size = np.bincount(component_id) # questions in each component

    rng = np.random.default_rng(seed)
    test_components = []
    n_test = 0
    for component in rng.permutation(number_of_components):
        if n_test >= test_fraction * n:
            break
        test_components.append(component) # this component, with ALL its pairs, goes to test
        n_test += component_size[component]

    questions = questions.copy()
    df_clean = df_clean.copy()
    questions["component_id"] = component_id
    questions["split"] = "dev"  # everybody starts in dev
    questions.loc[questions["component_id"].isin(test_components), "split"] = "test"

    split_of_question = questions.set_index("qid")["split"] 
    df_clean["split"] = df_clean["qid1"].map(split_of_question) 

    target = df_clean.pop("is_duplicate")
    df_clean["is_duplicate"] = target

    print(f"Components: {number_of_components}")
    print(df_clean["split"].value_counts())
    return questions, df_clean

questions, df_clean = split_dev_test(df_clean, questions)

Components: 200335
split
dev     323431
test     80341
Name: count, dtype: int64


# Saving tables

In [15]:
def save_tables(questions, df_clean, out_dir):
    """Saves tables in files"""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    questions_out = questions[["qid", "text_norm", "group_id", "split"]]
    pairs_out = df_clean[["qid1", "qid2", "conflicting_pair", "split", "is_duplicate"]]

    questions_out.to_parquet(out_dir / "questions.parquet", index=False)
    pairs_out.to_parquet(out_dir / "pairs.parquet", index=False)

    q_back = pd.read_parquet(out_dir / "questions.parquet")
    p_back = pd.read_parquet(out_dir / "pairs.parquet")
    print(f"questions: {q_back.shape}, pairs: {p_back.shape}")
    assert q_back.equals(questions_out) and p_back.equals(pairs_out)

save_tables(questions, df_clean, "../data/processed")

questions: (537174, 4), pairs: (403772, 5)


In [18]:
#some checks to rememeber
used_questions = pd.concat([df_clean["qid1"], df_clean["qid2"]]).unique()
unused_questions = ~df_unique_questions.index.isin(used_questions)
print(f"Questions not used in any pair: {unused_questions.sum()}")

print(df_clean.is_duplicate.value_counts())
baseline_acc = len(df_clean[df_clean.is_duplicate == 0]) / len(df_clean)
print(f"Baseline Accuracy: {baseline_acc:.2f}")
print(f"Duplicate Pairs: {1 - baseline_acc:.2f}")

Questions not used in any pair: 25
is_duplicate
0    254999
1    148773
Name: count, dtype: int64
Baseline Accuracy: 0.63
Duplicate Pairs: 0.37
